# TSAD on Kaggle (GPU)

Same pipeline as Colab, Kaggle paths. Attach the repo as a dataset (or upload `repo.zip`
as input), enable GPU (Settings → Accelerator → GPU T4 x2). Outputs persist to `/kaggle/working/`
— download them or `Save Version` (Output) before session end (9 h limit / 20 h weekly).

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
OUT = '/kaggle/working/tsad_runs'
!mkdir -p {OUT}

In [ ]:
import os, glob
REPO = '/kaggle/working/tsad'
cands = glob.glob('/kaggle/input/*/surya-1') + glob.glob('/kaggle/input/*')
print('inputs:', cands)
import shutil
src = next((c for c in cands if os.path.isdir(os.path.join(c, 'src', 'tsad'))), None)
zipc = next((c for c in glob.glob('/kaggle/input/*/*.zip') + glob.glob('/kaggle/input/*.zip')), None)
assert src or zipc, 'attach repo dataset or repo.zip'
if os.path.exists(REPO): shutil.rmtree(REPO)
if src: shutil.copytree(src, REPO)
else:
    !unzip -q {zipc} -d /kaggle/working && mv /kaggle/working/surya-1 {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

In [ ]:
!python -m pytest -q 2>&1 | tail -2
!python scripts/data_summary.py --config configs/smoke.yaml 2>&1 | tail -1

In [ ]:
import subprocess
SEEDS = '0 1 2'  # paper: '0 1 2 3 4' + configs/default.yaml
for exp in ['e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8']:
    import os
    if os.path.exists(f'reports/exp/{exp}.csv'):
        print(exp, 'exists -> skip'); continue
    print('== running', exp, '==', flush=True)
    subprocess.run(['python', 'scripts/run_experiment.py', '--config', 'configs/fast.yaml',
                      '--exp', exp, '--seeds'] + SEEDS.split() + ['--outdir', 'reports/exp'],
                     check=True)
    subprocess.run(['cp', '-r', 'reports', OUT + '/'], check=True)

In [ ]:
!python scripts/make_explanations.py --config configs/fast.yaml --opt seed=0 --out reports/explain_fast 2>&1 | tail -1
!python scripts/adapt.py --config configs/fast.yaml --epochs 5 --out reports/adapt_fast 2>&1 | tail -2
!cp -r reports {OUT}/ && echo synced — then Save Version / download {OUT}

## Timeouts & resume

- Kaggle sessions cap at ~9 h (GPU 20 h/week). Rerun the loop cell to resume (CSV markers skip
  finished exps). Internet is off by default — enable it for pip (Settings → Internet → On), or
  pre-install deps into a custom image. See `docs/GPU_RUNS.md` for the runtime table.